In [6]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================

INPUT_DIR = r"C:\Users\ashut\BE_Project\Dataset\Final_Merged_CF_Renumbered"
OUTPUT_DIR = r"C:\Users\ashut\BE_Project\Dataset\Generated_FAF_CNN_Dataset"

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==========================================
# 2. THE GENERATOR ARCHITECTURE
# ==========================================

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, act="relu", use_dropout=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False, padding_mode="reflect")
            if down
            else nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU() if act == "relu" else nn.LeakyReLU(0.2),
        )
        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.use_dropout else x
    
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, features=48): 
        super().__init__()
        self.initial_down = nn.Sequential(
            nn.Conv2d(in_channels, features, 4, 2, 1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )
        self.down1 = Block(features, features * 2, down=True, act="leaky")
        self.down2 = Block(features * 2, features * 4, down=True, act="leaky")
        self.down3 = Block(features * 4, features * 8, down=True, act="leaky")
        self.down4 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down5 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down6 = Block(features * 8, features * 8, down=True, act="leaky")
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features * 8, features * 8, 4, 2, 1, padding_mode="reflect"),
            nn.ReLU()
        )
        self.up1 = Block(features * 8, features * 8, down=False, act="relu", use_dropout=True)
        self.up2 = Block(features * 8 * 2, features * 8, down=False, act="relu", use_dropout=True)
        self.up3 = Block(features * 8 * 2, features * 8, down=False, act="relu", use_dropout=True)
        self.up4 = Block(features * 8 * 2, features * 8, down=False, act="relu")
        self.up5 = Block(features * 8 * 2, features * 4, down=False, act="relu")
        self.up6 = Block(features * 4 * 2, features * 2, down=False, act="relu")
        self.up7 = Block(features * 2 * 2, features, down=False, act="relu")
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features * 2, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    def forward(self, x):
        d1 = self.initial_down(x)
        d2 = self.down1(d1)
        d3 = self.down2(d2)
        d4 = self.down3(d3)
        d5 = self.down4(d4)
        d6 = self.down5(d5)
        d7 = self.down6(d6)
        bottleneck = self.bottleneck(d7)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d7], 1))
        up3 = self.up3(torch.cat([up2, d6], 1))
        up4 = self.up4(torch.cat([up3, d5], 1))
        up5 = self.up5(torch.cat([up4, d4], 1))
        up6 = self.up6(torch.cat([up5, d3], 1))
        up7 = self.up7(torch.cat([up6, d2], 1))
        return self.final_up(torch.cat([up7, d1], 1))


# ==========================================
# 2.5 LOAD YOUR TRAINED GENERATOR WEIGHTS
# ==========================================
print("Loading Generator Weights...")

# 1. Instantiate the model
G_AB = UNetGenerator().to(device)

# 2. Point to your specific weights file
weights_path = r"C:\Users\ashut\BE_Project\cycleGAN_Weights\G_AB_95.pth"

# 3. Load the weights into the model
G_AB.load_state_dict(torch.load(weights_path, map_location=device))

# 4. VERY IMPORTANT: Set the model to evaluation mode!
# This stops Dropout and BatchNorm layers from behaving like they are still training.
G_AB.eval() 

print("Weights loaded successfully!")

# ==========================================
# 3. TRANSFORMS
# ==========================================

img_transform = transforms.Compose([
        transforms.Resize((256, 256), Image.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

# ==========================================
# 4. THE BATCH CONVERSION LOOP
# ==========================================
print("Starting mass conversion...")
count = 0

# os.walk traverses your directory and subdirectories
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            
            # 1. Get exact paths
            cf_img_path = os.path.join(root, file)
            
            # Figure out which class folder this is (e.g., 'AMD', 'Normal')
            class_name = os.path.basename(root) 
            
            # Create the matching target directory (e.g., OUTPUT_DIR/AMD)
            target_class_dir = os.path.join(OUTPUT_DIR, class_name)
            os.makedirs(target_class_dir, exist_ok=True)
            
            # 2. Load and transform the image
            img = Image.open(cf_img_path).convert('RGB')
            img_tensor = img_transform(img).unsqueeze(0).to(device) 
            
            # 3. Generate the FAF image using your G_AB model!
            with torch.no_grad(): 
                fake_faf_tensor = G_AB(img_tensor) # <--- THIS is where the magic happens
                
            # 4. Save the generated image
            new_filename = file.replace('CF_', 'GenFAF_')
            save_path = os.path.join(target_class_dir, new_filename)
            
            # save_image automatically handles denormalizing from [-1, 1] back to [0, 1]
            save_image(fake_faf_tensor, save_path, normalize=True, value_range=(-1, 1))
            
            count += 1
            if count % 25 == 0:
                print(f"Processed {count} images...")

print(f"\n✅ Conversion Complete! {count} FAF images generated and sorted in: {OUTPUT_DIR}")

Using device: cuda
Loading Generator Weights...


C:\Users\ashut\AppData\Local\Temp\ipykernel_5464\337600629.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  G_AB.load_state_dict(torch.load(weights_path, map_location=de

Weights loaded successfully!
Starting mass conversion...
Processed 25 images...
Processed 50 images...
Processed 75 images...
Processed 100 images...
Processed 125 images...
Processed 150 images...
Processed 175 images...
Processed 200 images...
Processed 225 images...
Processed 250 images...

✅ Conversion Complete! 273 FAF images generated and sorted in: C:\Users\ashut\BE_Project\Dataset\Generated_FAF_CNN_Dataset
